In [45]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/all-data-csv/All_data.csv


In [46]:
pip install pandas torch transformers datasets evaluate seqeval kagglehub

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


# Libraries:

In [47]:
# Libraries:

import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback
)
import evaluate

In [48]:
# Data:

df = pd.read_excel('/kaggle/input/all-data-csv/All_data.csv')

/usr/local/lib/python3.11/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [49]:
# Preprocessing:

sentences = []
current_tokens = []
current_tags = []

for _, row in df.iterrows():
    word = str(row['word']).strip() if pd.notna(row['word']) else ''
    aspect = str(row['aspect']).strip() if pd.notna(row['aspect']) else 'O'
    category = str(row['categort']).strip() if pd.notna(row.get('categort', 'O')) else 'O'
    
    if word:
        current_tokens.append(word)
        
        if aspect == 'B' and category == 'O':
            current_tags.append('O')
        elif aspect != 'O':
            current_tags.append(f"{aspect}-{category}")
        else:
            current_tags.append('O')
    
    if pd.isna(row['Sentence #']) or str(row['Sentence #']).startswith('Sentence:'):
        if current_tokens:
            sentences.append({'tokens': current_tokens, 'ner_tags': current_tags})
        current_tokens = []
        current_tags = []

if current_tokens:
    sentences.append({'tokens': current_tokens, 'ner_tags': current_tags})

In [50]:
# Split Data:

train_sentences, eval_sentences = train_test_split(
    sentences, 
    test_size=0.2,
    random_state=42,
    shuffle=True
)

raw_datasets = DatasetDict({
    'train': Dataset.from_list(train_sentences),
    'eval': Dataset.from_list(eval_sentences)
})

In [51]:
# Model:

model_checkpoint = "UBC-NLP/MARBERT"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

label_names = ['O', 'B-Service', 'I-Service', 'B-Price', 'I-Price', 
               'B-Time', 'I-Time', 'B-Location', 'I-Location']

id2label = {i: label for i, label in enumerate(label_names)}
label2id = {v: k for k, v in id2label.items()}

In [52]:
# Tokenization:

def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id != current_word:
            current_word = word_id
            label = -100 if word_id is None else label2id[labels[word_id]]
            new_labels.append(label)
        elif word_id is None:
            new_labels.append(-100)
        else:
            label = label2id[labels[word_id]]
            if label % 2 == 1:
                label += 1
            new_labels.append(label)
    return new_labels

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], 
        truncation=True, 
        is_split_into_words=True,
        max_length=128,
        padding='max_length'
    )
    all_labels = examples["ner_tags"]
    new_labels = []
    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(i)
        new_labels.append(align_labels_with_tokens(labels, word_ids))
    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs

tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

Map:   0%|          | 0/83624 [00:00<?, ? examples/s]

Map:   0%|          | 0/20907 [00:00<?, ? examples/s]

In [53]:
# Metrics:

metric = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [54]:
# Model Pretraining:

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id,
    num_labels=len(label_names)
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at UBC-NLP/MARBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [55]:
# Data Collator:

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [56]:
# Training Arguments:

args = TrainingArguments(
    "marbert-absa-finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to="none",
)


# Training:

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["eval"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.516100,0.484417,0.939693,0.667284,0.780400,0.879031
2,0.448200,0.514147,0.903985,0.695903,0.786412,0.881728
3,0.408000,0.531995,0.873443,0.721845,0.790441,0.879587


TrainOutput(global_step=31359, training_loss=0.46307333073995716, metrics={'train_runtime': 4524.4624, 'train_samples_per_second': 55.448, 'train_steps_per_second': 6.931, 'total_flos': 1.6389047129942016e+16, 'train_loss': 0.46307333073995716, 'epoch': 3.0})

In [57]:
# Final Evaluation
results = trainer.evaluate()
print("Results:")
print(f"Accuracy: {results['eval_accuracy']:.4f}")
print(f"F1 Score: {results['eval_f1']:.4f}")
print(f"Precision: {results['eval_precision']:.4f}")
print(f"Recall: {results['eval_recall']:.4f}")

Results:
Accuracy: 0.8796
F1 Score: 0.7904
Precision: 0.8734
Recall: 0.7218
